In [1]:
import librosa
from pesq import pesq

#Оценка качества звука
def calculate_pesq(reference_file, cleaned_file, target_sr=16000):
    """
    4.0-4.5  Отличное качество
    3.0-4.0  Хорошее качество
    2.0-3.0  Удовлетворительное качество
    1.0-2.0  Плохое качество
    <1.0     Очень плохое качество 
    """
    try:
        ref_audio, sr_ref = librosa.load(reference_file, sr=target_sr)
        clean_audio, sr_deg = librosa.load(cleaned_file, sr=target_sr)
        
        min_len = min(len(ref_audio), len(clean_audio))
        ref_audio = ref_audio[:min_len]
        clean_audio = clean_audio[:min_len]
        
        mode = 'wb' if target_sr == 16000 else 'nb'
        
        # Вычисляем PESQ
        score = pesq(target_sr, ref_audio, clean_audio, mode)
        return score
        
    except Exception as e:
        print(f"Ошибка: {e}")
        return None

In [2]:
import librosa
import soundfile as sf
from pystoi import stoi
import numpy as np

#Оценка понятности речи 
def calculate_stoi(path_clean, path_denoised, target_sr=16000):
    """
    1.0   Идеальная разборчивость.
    0.75  Хорошая разборчивость.
    0.5   Средняя разборчивость, слова могут быть неразборчивы в шуме
    0.0   Полная неразборчивость.
    """
    try:
        clean_speech, fs = librosa.load(path_clean, sr=target_sr, mono=True)
        denoised_speech, _ = librosa.load(path_denoised, sr=target_sr, mono=True)

        min_len = min(len(clean_speech), len(denoised_speech))
        clean_speech = clean_speech[:min_len]
        denoised_speech = denoised_speech[:min_len]

        stoi_score = stoi(clean_speech, denoised_speech, fs, extended=False)
        return stoi_score
        
    except Exception as e:
        print(f"Ошибка: {e}")
        return None

In [ ]:
from pathlib import Path
import numpy as np

def evaluate_folders(
    sota_folder="cleaned_results_sota",
    method_folder="cleaned_results",
    target_sr=16000,
):
    supported = {'.wav', '.flac', '.ogg', '.mp3'}
    SOTA_PREFIX   = 'enhanced_'
    METHOD_PREFIX = 'cleaned_'

    def bare(stem, prefix):
        return stem[len(prefix):] if stem.startswith(prefix) else stem

    # Индексируем SOTA по «голому» имени (без enhanced_)
    sota_files = {
        bare(f.stem, SOTA_PREFIX): f
        for f in Path(sota_folder).rglob('*')
        if f.suffix.lower() in supported
    }

    method_files = sorted(
        f for f in Path(method_folder).rglob('*')
        if f.suffix.lower() in supported
    )

    print(f"\n{'='*60}")
    print(f"  SOTA   : {sota_folder}  (префикс '{SOTA_PREFIX}')")
    print(f"  Метод  : {method_folder}  (префикс '{METHOD_PREFIX}')")
    print(f"{'='*60}\n")

    all_pesq, all_stoi = [], []

    for method_path in method_files:
        key = bare(method_path.stem, METHOD_PREFIX)   # "test_1", "test1_clear" …

        if key not in sota_files:
            print(f"  [~] Пара для '{method_path.name}' не найдена, пропускаем\n")
            continue

        sota_path = sota_files[key]
        print(f"{method_path.name}")
        print(f"SOTA-референс : {sota_path.name}")

        pesq_score = calculate_pesq(str(sota_path), str(method_path), target_sr)
        stoi_score = calculate_stoi(str(sota_path), str(method_path), target_sr)

        if pesq_score is not None:
            all_pesq.append(pesq_score)
            print(f"    PESQ : {pesq_score:.3f}")

        if stoi_score is not None:
            all_stoi.append(stoi_score)
            print(f"    STOI : {stoi_score:.3f}")

        print()

    # Итоговая сводка
    print(f"{'─'*60}")
    print(f"Итого файлов: {len(all_pesq)}")
    if all_pesq:
        print(f"  PESQ — avg: {np.mean(all_pesq):.3f}  "
              f"min: {np.min(all_pesq):.3f}  max: {np.max(all_pesq):.3f}")
    if all_stoi:
        print(f"  STOI — avg: {np.mean(all_stoi):.3f}  "
              f"min: {np.min(all_stoi):.3f}  max: {np.max(all_stoi):.3f}")
    print(f"{'─'*60}\n")

In [6]:
# if __name__ == "__main__":
#     # пока вручную вводим файлики
#     ref_path = "dataset/test1_clear.ogg" 
#     clean_path = "cleaned_results/cleaned_test1_clear.wav" 
    
#     score_pesq = calculate_pesq(ref_path, clean_path, target_sr=16000)
    
#     if score_pesq is not None:
#         print(f"Оценка PESQ: {score_pesq:.3f}")

#     score_stoi = calculate_stoi(ref_path, clean_path, target_sr=16000)
    
#     if score_stoi is not None:
#         print(f"Оценка STOI: {score_stoi:.3f}")

if __name__ == "__main__":
    evaluate_folders(
        sota_folder="cleaned_results_sota",
        method_folder="cleaned_results",
        target_sr=16000,
    )


  SOTA   : cleaned_results_sota  (префикс 'enhanced_')
  Метод  : cleaned_results  (префикс 'cleaned_')

  ▶ cleaned_test1_clear.wav
    SOTA-референс : enhanced_test1_clear.wav
    PESQ : 1.492
    STOI : 0.876

  ▶ cleaned_test1_musicSpeech.wav
    SOTA-референс : enhanced_test1_musicSpeech.wav
    PESQ : 1.548
    STOI : 0.837

  ▶ cleaned_test2_speech_and_music.wav
    SOTA-референс : enhanced_test2_speech_and_music.wav
    PESQ : 1.857
    STOI : 0.891

  ▶ cleaned_test2_speech_and_music_2.wav
    SOTA-референс : enhanced_test2_speech_and_music_2.wav
    PESQ : 1.682
    STOI : 0.900

  ▶ cleaned_test2_speechMore.wav
    SOTA-референс : enhanced_test2_speechMore.wav
    PESQ : 1.727
    STOI : 0.922

  ▶ cleaned_test3_musicMore.wav
    SOTA-референс : enhanced_test3_musicMore.wav
    PESQ : 1.531
    STOI : 0.667

  ▶ cleaned_test4_MaximMoment.wav
    SOTA-референс : enhanced_test4_MaximMoment.wav
    PESQ : 1.096
    STOI : 0.823

  [~] Пара для 'cleaned_test_1.wav' не найдена, 

C:\Users\chiki\AppData\Local\Temp\ipykernel_33428\1806334659.py:14: UserWarning: PySoundFile failed. Trying audioread instead.
  ref_audio, sr_ref = librosa.load(reference_file, sr=target_sr)
d:\BFU\kursach\.venv\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


    PESQ : 1.189
    STOI : 0.777

────────────────────────────────────────────────────────────
  Итого файлов: 8
  PESQ — avg: 1.515  min: 1.096  max: 1.857
  STOI — avg: 0.837  min: 0.667  max: 0.922
────────────────────────────────────────────────────────────



C:\Users\chiki\AppData\Local\Temp\ipykernel_33428\1566180251.py:15: UserWarning: PySoundFile failed. Trying audioread instead.
  clean_speech, fs = librosa.load(path_clean, sr=target_sr, mono=True)
